# SQL Sales & Business Analysis

This notebook uses SQL to analyze sales, profitability, customer value, discounting, order outcomes, and geographic performance for a fictional mid-sized e-commerce business.

The analysis is performed on the cleaned datasets generated during the data-cleaning stage of Project 2.

### Objectives

- Evaluate overall sales and profitability trends
- Identify category and product-level profitability differences
- Analyze customer value and profitability
- Examine the relationship between discount levels and profit
- Analyze returns and cancellations
- Compare business performance across states




## 1. Database Setup

The cleaned orders, customers, and products datasets are loaded into a SQLite database so that the analysis can be performed using SQL.

The database contains three tables:

- `orders`: order-level transactions and financial metrics
- `customers`: customer attributes and customer segments
- `products`: product attributes, pricing, and cost information

In [29]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
folder_path = '/content/drive/MyDrive/Colab Notebooks/Personal Projects/Project_2_sales_analysis/'

orders = pd.read_csv(folder_path+"orders_clean.csv")
customers = pd.read_csv(folder_path+"customers_clean.csv")
products = pd.read_csv(folder_path+"products_clean.csv")

orders["order_date"] = pd.to_datetime(orders["order_date"])
customers["customer_since"] = pd.to_datetime(customers["customer_since"])

orders.shape, customers.shape, products.shape

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


((30000, 17), (2500, 6), (72, 6))

In [30]:
import sqlite3
import os

db_path = "/content/drive/MyDrive/Colab Notebooks/Personal Projects/Project_2_sales_analysis/sales_analysis.db"

conn = sqlite3.connect(db_path)

print("Database path:", db_path)
print("Database exists:", os.path.exists(db_path))
print("Database size:", os.path.getsize(db_path), "bytes")

Database path: /content/drive/MyDrive/Colab Notebooks/Personal Projects/Project_2_sales_analysis/sales_analysis.db
Database exists: True
Database size: 4784128 bytes


In [31]:
orders.to_sql("orders", conn, if_exists="replace", index=False)
customers.to_sql("customers", conn, if_exists="replace", index=False)
products.to_sql("products", conn, if_exists="replace", index=False)

72

In [32]:
pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

,name
0,orders
1,customers
2,products


## 2. Annual Business Performance

The first objective is to assess overall business performance from 2023 to 2025.

This analysis examines order volume, revenue, profit, and profit margin to determine whether the business experienced meaningful changes in scale or profitability over the period.

### Annual Revenue and Profitability

In [33]:
query = """
SELECT
    strftime('%Y', order_date) AS year,
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(revenue) AS total_revenue,
    SUM(profit) AS total_profit,
    ROUND(
        SUM(profit) * 100.0 / SUM(revenue),
        2
    ) AS profit_margin
FROM orders
GROUP BY year
ORDER BY year;
"""

annual_sql = pd.read_sql(query, conn)

annual_sql

,year,total_orders,total_revenue,total_profit,profit_margin
0,2023,9937,3.673607e+08,9.047083e+07,24.63
1,2024,9951,3.647547e+08,8.961326e+07,24.57
2,2025,10112,3.699009e+08,9.115654e+07,24.64


**Interpretation:** Revenue remained broadly stable over the three-year period. Order volume increased modestly from 2023 to 2025, while overall profit margin remained close to 24.6%. The results do not indicate a meaningful change in overall business profitability during the period.

## 3. Category Profitability

The next objective is to compare profitability across product categories.

Revenue and profit are recorded at the order level, while category information is stored in the products table. A join between the two tables allows order-level financial metrics to be analyzed by product category.

In [34]:
query = """
SELECT
    p.category,
    SUM(o.revenue) AS total_revenue,
    SUM(o.profit) AS total_profit,
    ROUND(
        SUM(o.profit) * 100.0 / SUM(o.revenue),
        2
    ) AS profit_margin
FROM orders AS o
JOIN products AS p
    ON o.product_id = p.product_id
GROUP BY p.category
ORDER BY profit_margin DESC;
"""

category_sql = pd.read_sql(query, conn)

category_sql

,category,total_revenue,total_profit,profit_margin
0,Electronics,6.597231e+08,1.675399e+08,25.40
1,Home Appliances,1.741430e+08,4.379013e+07,25.15
2,Furniture,2.368701e+08,5.810293e+07,24.53
3,Office Supplies,3.127998e+07,1.807620e+06,5.78


**Interpretation:** Office Supplies has a substantially lower profit margin (5.8%) than the other categories, which range from 24.5% to 25.4%. The result indicates materially weaker profitability within the category and warrants further investigation at the subcategory and product levels.

### Office Supplies: Subcategory Profitability

Because Office Supplies has substantially lower profitability than the other categories, the next step is to examine its subcategories.

This helps determine whether the weakness is broad-based or concentrated in a specific part of the category.

In [35]:
query = """
SELECT
    p.subcategory,
    SUM(o.revenue) AS total_revenue,
    SUM(o.profit) AS total_profit,
    ROUND(
        SUM(o.profit) * 100.0 / SUM(o.revenue),
        2
    ) AS profit_margin
FROM orders AS o
JOIN products AS p
    ON o.product_id = p.product_id
WHERE p.category = 'Office Supplies'
GROUP BY p.subcategory
ORDER BY profit_margin;
"""

subcategory_sql = pd.read_sql(query, conn)

subcategory_sql

,subcategory,total_revenue,total_profit,profit_margin
0,Writing,7523575.92,-3.536602e+05,-4.70
1,Paper,11120209.27,9.208475e+05,8.28
2,Stationery,12636192.07,1.240433e+06,9.82


**Interpretation:** Profitability is weak across all three Office Supplies subcategories, with Writing being the only loss-making subcategory at -4.7% margin. Paper and Stationery remain profitable but at substantially lower margins than the other product categories. The results suggest that Office Supplies has structurally weaker economics, with Writing requiring particular attention.

### Product-Level Profitability Within Office Supplies

The subcategory analysis identified Writing as the only loss-making Office Supplies subcategory. The next step is to examine profitability at the individual product level and identify products generating negative profit.

In [36]:
query = """
SELECT
    p.product_id,
    p.product_name,
    p.subcategory,
    SUM(o.revenue) AS total_revenue,
    SUM(o.profit) AS total_profit,
    ROUND(
        SUM(o.profit) * 100.0 / SUM(o.revenue),
        2
    ) AS profit_margin
FROM orders AS o
JOIN products AS p
    ON o.product_id = p.product_id
WHERE p.category = 'Office Supplies'
GROUP BY
    p.product_id,
    p.product_name,
    p.subcategory
HAVING SUM(o.profit) < 0
ORDER BY total_profit;
"""

loss_making_products_sql = pd.read_sql(query, conn)

loss_making_products_sql

,product_id,product_name,subcategory,total_revenue,total_profit,profit_margin
0,P0069,Writing Product 3,Writing,602677.87,-261825.206240,-43.44
1,P0070,Writing Product 4,Writing,268049.04,-209314.712623,-78.09
2,P0072,Writing Product 6,Writing,600895.75,-196744.555693,-32.74
3,P0068,Writing Product 2,Writing,571745.23,-105843.271243,-18.51
4,P0058,Stationery Product 4,Stationery,1522032.31,-74437.131607,-4.89
5,P0066,Paper Product 6,Paper,1113515.03,-64425.959391,-5.79


**Interpretation:** Six Office Supplies products generate negative profit, with four belonging to the Writing subcategory. The largest losses are concentrated among the Writing products, indicating that the category-level profitability issue is driven by a relatively small group of products rather than being evenly distributed across all Office Supplies products.

## 4. Customer Profitability

The next objective is to understand customer-level economics.

Customer attributes such as segment are stored in the customers table, while revenue and profit are recorded at the order level. Aggregating orders by customer allows the analysis to identify differences in customer value and profitability.

### Customer-Level Economics

In [37]:
query = """
SELECT
    o.customer_id,
    c.customer_segment,
    COUNT(DISTINCT o.order_id) AS total_orders,
    SUM(o.revenue) AS total_revenue,
    SUM(o.profit) AS total_profit,
    ROUND(
        SUM(o.profit) * 100.0 / SUM(o.revenue),
        2
    ) AS profit_margin
FROM orders AS o
JOIN customers AS c
    ON o.customer_id = c.customer_id
GROUP BY
    o.customer_id,
    c.customer_segment
ORDER BY total_profit DESC;
"""

customer_sql = pd.read_sql(query, conn)

customer_sql.head(10)

,customer_id,customer_segment,total_orders,total_revenue,total_profit,profit_margin
0,C02094,Consumer,14,2759153.90,841328.622363,30.49
1,C01131,Small Business,13,1416994.79,411320.114689,29.03
2,C01357,Enterprise,20,1277204.43,403128.991143,31.56
3,C00427,Consumer,10,1341907.03,389684.858907,29.04
4,C01434,Small Business,17,1122862.29,372706.362074,33.19
5,C01049,Small Business,21,1385942.38,370278.276006,26.72
6,C00046,Small Business,14,1173661.03,368232.664463,31.37
7,C00212,Small Business,16,1147564.62,363924.742645,31.71
8,C00004,Consumer,14,1270399.52,363459.157133,28.61
9,C00787,Consumer,18,1290691.42,359823.286645,27.88


**Interpretation:** Customer-level profitability varies across the customer base, with differences in total revenue and profit reflecting differences in order activity and customer value. Overall customer margins remain relatively consistent, with a median customer profit margin of approximately 24.4% based on the broader customer-level analysis.

### Customer Profit Concentration

To assess whether the business is heavily dependent on a small group of high-value customers, customers are ranked by cumulative profit. The analysis then measures the share of total customer profit generated by the top 10% of customers.

In [38]:
query = """
WITH customer_profit AS (
    SELECT
        customer_id,
        SUM(profit) AS total_profit
    FROM orders
    GROUP BY customer_id
),

ranked_customers AS (
    SELECT
        customer_id,
        total_profit,
        NTILE(10) OVER (
            ORDER BY total_profit DESC
        ) AS profit_decile
    FROM customer_profit
)

SELECT
    ROUND(
        SUM(
            CASE
                WHEN profit_decile = 1
                THEN total_profit
                ELSE 0
            END
        ) * 100.0
        / SUM(total_profit),
        2
    ) AS top_10_profit_share
FROM ranked_customers;
"""

profit_concentration_sql = pd.read_sql(query, conn)

profit_concentration_sql

,top_10_profit_share
0,21.21


**Interpretation:** The top 10% of customers by cumulative profit contribute approximately 21.2% of total customer profit. This indicates moderate customer concentration rather than heavy dependence on a small group of high-value customers.

### Customer Segment Economics

Customer profitability can also be examined by customer segment to determine whether Consumer, Small Business, and Enterprise customers generate materially different levels of revenue or profit.

In [39]:
query = """
SELECT
    c.customer_segment,
    COUNT(DISTINCT c.customer_id) AS customers,
    SUM(o.revenue) AS total_revenue,
    SUM(o.profit) AS total_profit,
    ROUND(
        SUM(o.revenue) * 1.0
        / COUNT(DISTINCT c.customer_id),
        2
    ) AS revenue_per_customer,
    ROUND(
        SUM(o.profit) * 1.0
        / COUNT(DISTINCT c.customer_id),
        2
    ) AS profit_per_customer,
    ROUND(
        SUM(o.profit) * 100.0
        / SUM(o.revenue),
        2
    ) AS profit_margin
FROM orders AS o
JOIN customers AS c
    ON o.customer_id = c.customer_id
GROUP BY c.customer_segment
ORDER BY profit_per_customer DESC;
"""

segment_sql = pd.read_sql(query, conn)

segment_sql

,customer_segment,customers,total_revenue,total_profit,revenue_per_customer,profit_per_customer,profit_margin
0,Consumer,1486,6.739076e+08,1.662759e+08,453504.43,111894.97,24.67
1,Small Business,769,3.247197e+08,7.978339e+07,422262.31,103749.54,24.57
2,Enterprise,245,1.033889e+08,2.518131e+07,421995.55,102780.85,24.36


**Interpretation:** Consumer customers generate slightly higher revenue and profit per customer than Small Business and Enterprise customers. However, profit margins are very similar across all three segments, ranging from approximately 24.4% to 24.7%. This suggests that differences in total revenue and profit are driven primarily by the size of each customer group rather than materially different profitability.

## 5. Discounting & Profitability

The next objective is to examine how profitability varies across different discount levels.

The analysis compares order volume, revenue, profit, and profit margin across observed discount levels. This helps assess whether deeper discounts are associated with materially lower profitability.

### Profitability by Discount Level

In [40]:
query = """
SELECT
    ROUND(discount_pct * 100, 0) AS discount_level,
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(revenue) AS total_revenue,
    SUM(profit) AS total_profit,
    ROUND(
        SUM(profit) * 100.0 / SUM(revenue),
        2
    ) AS profit_margin,
    ROUND(
        SUM(revenue) * 1.0 / COUNT(DISTINCT order_id),
        2
    ) AS revenue_per_order,
    ROUND(
        SUM(profit) * 1.0 / COUNT(DISTINCT order_id),
        2
    ) AS profit_per_order
FROM orders
GROUP BY discount_pct
ORDER BY discount_pct;
"""

discount_sql = pd.read_sql(query, conn)

discount_sql

,discount_level,total_orders,total_revenue,total_profit,profit_margin,revenue_per_order,profit_per_order
0,0.0,6086,2.451280e+08,7.732102e+07,31.54,40277.36,12704.74
1,5.0,7529,2.848295e+08,7.895734e+07,27.72,37830.98,10487.10
2,10.0,7489,2.783868e+08,6.647237e+07,23.88,37172.76,8876.00
3,15.0,4422,1.508509e+08,2.966317e+07,19.66,34113.72,6708.09
4,20.0,3035,1.009234e+08,1.506887e+07,14.93,33253.19,4965.03
5,25.0,1439,4.189762e+07,3.757867e+06,8.97,29115.79,2611.44


**Interpretation:** Profit margin declines consistently as discount levels increase, falling from 31.5% with no discount to 9.0% at a 25% discount. Revenue per order also declines from approximately ₹40.3K to ₹29.1K, while profit per order falls more sharply from ₹12.7K to ₹2.6K. The results show a clear association between deeper discounting and lower profitability, although the analysis does not establish causality.

## 6. Returns & Cancellations

The next objective is to examine order outcomes and identify whether returns or cancellations are concentrated in particular product categories.

Because the dataset does not contain refund amounts, returned and cancelled order values are treated as gross order value rather than realized financial loss.

### Order Outcomes by Category

In [41]:
query = """
SELECT
    p.category,
    COUNT(DISTINCT o.order_id) AS total_orders,
    SUM(
        CASE
            WHEN o.order_status = 'Completed' THEN 1
            ELSE 0
        END
    ) AS completed_orders,
    SUM(
        CASE
            WHEN o.order_status = 'Returned' THEN 1
            ELSE 0
        END
    ) AS returned_orders,
    SUM(
        CASE
            WHEN o.order_status = 'Cancelled' THEN 1
            ELSE 0
        END
    ) AS cancelled_orders,
    ROUND(
        SUM(
            CASE
                WHEN o.order_status = 'Returned' THEN 1
                ELSE 0
            END
        ) * 100.0 / COUNT(DISTINCT o.order_id),
        2
    ) AS return_rate,
    ROUND(
        SUM(
            CASE
                WHEN o.order_status = 'Cancelled' THEN 1
                ELSE 0
            END
        ) * 100.0 / COUNT(DISTINCT o.order_id),
        2
    ) AS cancellation_rate
FROM orders AS o
JOIN products AS p
    ON o.product_id = p.product_id
GROUP BY p.category
ORDER BY return_rate DESC;
"""

category_outcomes_sql = pd.read_sql(query, conn)

category_outcomes_sql

,category,total_orders,completed_orders,returned_orders,cancelled_orders,return_rate,cancellation_rate
0,Home Appliances,7421,6643,574,204,7.73,2.75
1,Furniture,7591,6813,522,256,6.88,3.37
2,Office Supplies,7424,6706,508,210,6.84,2.83
3,Electronics,7564,6846,498,220,6.58,2.91


**Interpretation:** Return rates are broadly similar across categories, ranging from 6.6% to 7.7%. Home Appliances has the highest return rate at 7.7%, while Electronics has the lowest at 6.6%. Cancellation rates are also relatively consistent, ranging from 2.7% to 3.4%. The differences do not indicate a large category-level disparity in order outcomes, although Home Appliances may warrant further investigation.

### Order Outcomes by Sales Channel

Order outcomes are also compared across sales channels to determine whether return or cancellation rates differ materially between Online and Store orders.

In [42]:
query = """
SELECT
    sales_channel,
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(
        CASE
            WHEN order_status = 'Completed' THEN 1
            ELSE 0
        END
    ) AS completed_orders,
    SUM(
        CASE
            WHEN order_status = 'Returned' THEN 1
            ELSE 0
        END
    ) AS returned_orders,
    SUM(
        CASE
            WHEN order_status = 'Cancelled' THEN 1
            ELSE 0
        END
    ) AS cancelled_orders,
    ROUND(
        SUM(
            CASE
                WHEN order_status = 'Returned' THEN 1
                ELSE 0
            END
        ) * 100.0 / COUNT(DISTINCT order_id),
        2
    ) AS return_rate,
    ROUND(
        SUM(
            CASE
                WHEN order_status = 'Cancelled' THEN 1
                ELSE 0
            END
        ) * 100.0 / COUNT(DISTINCT order_id),
        2
    ) AS cancellation_rate
FROM orders
GROUP BY sales_channel
ORDER BY sales_channel;
"""

channel_outcomes_sql = pd.read_sql(query, conn)

channel_outcomes_sql

,sales_channel,total_orders,completed_orders,returned_orders,cancelled_orders,return_rate,cancellation_rate
0,Online,21101,19005,1479,617,7.01,2.92
1,Store,8899,8003,623,273,7.00,3.07


**Interpretation:** Return rates are virtually identical across Online and Store orders, at approximately 7.0% for both channels. Cancellation rates are also similar, at 2.9% Online and 3.1% in Store. This suggests that order outcomes are not materially different by sales channel in this dataset.

## 7. Geographic Performance

The final analysis examines revenue and customer distribution across states to identify meaningful geographic differences in business performance.

Total revenue reflects the size of each state's customer base, while revenue per customer provides a measure of average customer value.

### Revenue and Customer Distribution by State

In [43]:
query = """
SELECT
    c.state,
    COUNT(DISTINCT c.customer_id) AS customers,
    SUM(o.revenue) AS total_revenue,
    SUM(o.profit) AS total_profit,
    ROUND(
        SUM(o.revenue) * 1.0
        / COUNT(DISTINCT c.customer_id),
        2
    ) AS revenue_per_customer
FROM orders AS o
JOIN customers AS c
    ON o.customer_id = c.customer_id
GROUP BY c.state
ORDER BY total_revenue DESC;
"""

state_sql = pd.read_sql(query, conn)

state_sql

,state,customers,total_revenue,total_profit,revenue_per_customer
0,Maharashtra,500,2.150305e+08,5.293467e+07,430061.09
1,Tamil Nadu,283,1.256317e+08,3.091327e+07,443928.19
2,West Bengal,255,1.199656e+08,2.945591e+07,470453.41
3,Kerala,253,1.125499e+08,2.778237e+07,444861.38
4,Gujarat,250,1.121993e+08,2.772200e+07,448797.19
5,Rajasthan,249,1.084389e+08,2.674069e+07,435497.46
6,Telangana,251,1.061211e+08,2.617326e+07,422793.18
7,Karnataka,233,1.047025e+08,2.567347e+07,449367.05
8,Delhi,226,9.737666e+07,2.384500e+07,430870.19


**Interpretation:** Maharashtra generates the highest total revenue, primarily because it has the largest customer base. Revenue per customer varies across states, ranging from approximately ₹422.8K to ₹470.5K. Overall, geographic differences in customer value are moderate, suggesting that market size is a more important driver of total state revenue than large differences in individual customer value.

## 8. Key Business Findings

The SQL analysis reinforces several findings identified during the broader business analysis:

- Overall revenue and profitability remained broadly stable from 2023 to 2025.
- Office Supplies has substantially weaker profitability than the other product categories, with six loss-making products.
- Four of the six loss-making Office Supplies products belong to the Writing subcategory.
- The top 10% of customers by cumulative profit contribute approximately 21.2% of total customer profit.
- Higher discount levels are consistently associated with lower profit margins and lower profit per order.
- Return and cancellation rates are broadly similar across categories and sales channels.
- Maharashtra generates the highest total revenue because of its larger customer base, while differences in revenue per customer across states are moderate.

This SQL analysis demonstrates how relational data can be combined and aggregated to answer practical business questions using joins, conditional logic, common table expressions, and window functions.

In [44]:
conn.close()